# 032: Python Lists — Complete A–Z Method Reference, CPython Internals, and Production Engineering

## 1. WHAT IS A PYTHON LIST?

A Python `list` is an **ordered, mutable, heterogeneous** sequence type. Internally in CPython, it is implemented as a **dynamic array of pointers** (PyObject references), not a linked list.

### CPython Internal Representation (`listobject.h`)
```c
typedef struct {
    PyObject_VAR_HEAD        // ob_refcnt, ob_type, ob_size (length)
    PyObject **ob_item;      // Pointer to array of PyObject* pointers
    Py_ssize_t allocated;    // Number of slots allocated (>= ob_size)
} PyListObject;
```
- `ob_item`: A C array of `PyObject*` pointers. Each slot holds a reference to an arbitrary Python object.
- `ob_size`: The current number of elements in the list (what `len()` returns).
- `allocated`: The total number of slots available. When `ob_size == allocated`, the next append triggers a resize.

### Memory Layout
- An empty list on a 64-bit system consumes **56 bytes** (object header + pointer to the items array + allocated count).
- Each additional element adds **8 bytes** (one 64-bit pointer).
- The actual Python objects the pointers reference live elsewhere on the heap.

### Over-Allocation Growth Formula (`list_resize` in `listobject.c`)
$$\text{new\_allocated} = \text{newsize} + (\text{newsize} \gg 3) + (\text{newsize} < 9\ ?\ 3 : 6)$$
This yields growth factors of approximately 1.125x, producing the allocation pattern: 0 → 4 → 8 → 16 → 25 → 35 → 46 → 58 → 72 → 88.
The amortized cost of `append()` is therefore $O(1)$.

---


In [ ]:
import sys

# ── Tracking CPython list memory resizing ──
print("=" * 70)
print("SECTION 1: CPython List Memory Resizing Mechanics")
print("=" * 70)

lst = []
prev_size = sys.getsizeof(lst)
print(f"{'Length':>6} | {'Bytes':>6} | {'Alloc Slots':>11} | Event")
print("-" * 50)
print(f"{'0':>6} | {prev_size:>6} | {'0':>11} | Initial empty list")

for i in range(30):
    lst.append(i)
    curr_size = sys.getsizeof(lst)
    alloc_slots = (curr_size - 56) // 8  # 56 = base size, 8 = pointer size
    if curr_size != prev_size:
        print(f"{len(lst):>6} | {curr_size:>6} | {alloc_slots:>11} | RESIZE triggered")
        prev_size = curr_size



---
## 2. COMPLETE A–Z METHOD REFERENCE

Every built-in list method is documented below with:
- **Syntax** and **return value**
- **CPython internal behavior**
- **Time complexity** and **space complexity**
- **Edge cases**
- **Code examples** (basic → advanced → edge case)

---

### 2.1 `list.append(x)`
- **What it does**: Adds element `x` to the **end** of the list.
- **Returns**: `None` (in-place mutation).
- **CPython**: Calls `list_resize()` if `ob_size == allocated`. Otherwise, places the pointer at `ob_item[ob_size]` and increments `ob_size`.
- **Time**: $O(1)$ amortized. Occasional $O(n)$ when resize triggers a full copy.
- **Space**: $O(1)$ amortized (over-allocation absorbs growth).


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.append(x)")
print("=" * 70)

# Basic usage
fruits = ["apple", "banana"]
fruits.append("cherry")
print(f"After append('cherry'):    {fruits}")

# Appending different types (heterogeneous)
fruits.append(42)
fruits.append([1, 2, 3])  # Appends the entire list as ONE element
print(f"After append(42) & list:   {fruits}")

# Edge case: append None
fruits.append(None)
print(f"After append(None):        {fruits}")

# Edge case: append to empty list
empty = []
empty.append("first")
print(f"Append to empty list:      {empty}")

# Common mistake: append vs extend
combo = [1, 2]
combo.append([3, 4])  # This creates a nested list!
print(f"append([3,4]) creates:     {combo}  ← nested, NOT [1,2,3,4]")



### 2.2 `list.extend(iterable)`
- **What it does**: Appends each element from `iterable` to the list individually.
- **Returns**: `None` (in-place).
- **CPython**: Iterates over the iterable, calling `list_resize()` once for the total new size, then copies pointers.
- **Time**: $O(k)$ where $k$ = length of the iterable.
- **Space**: $O(k)$ for the new pointer slots.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.extend(iterable)")
print("=" * 70)

# Basic
a = [1, 2, 3]
a.extend([4, 5, 6])
print(f"extend([4,5,6]):           {a}")

# Extend with a tuple
a.extend((7, 8))
print(f"extend((7,8)):             {a}")

# Extend with a string (each character becomes an element!)
chars = []
chars.extend("hello")
print(f"extend('hello'):           {chars}  ← each char is separate")

# Extend with a generator
a.extend(x*x for x in range(3))
print(f"extend(generator):         {a}")

# Edge case: extend with empty iterable
a.extend([])
print(f"extend([]) — no change:    {a}")

# Difference from += operator (they are equivalent)
b = [1, 2]
b += [3, 4]  # Same as b.extend([3, 4])
print(f"+= operator result:        {b}")



### 2.3 `list.insert(i, x)`
- **What it does**: Inserts element `x` at position `i`. All elements at index ≥ `i` shift right by one.
- **Returns**: `None`.
- **CPython**: Calls `list_resize()` if needed, then uses `memmove()` to shift elements.
- **Time**: $O(n)$ — all elements after index `i` must be shifted.
- **Space**: $O(1)$ amortized.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.insert(i, x)")
print("=" * 70)

nums = [10, 20, 30, 40, 50]
nums.insert(2, 25)
print(f"insert(2, 25):             {nums}")

# Insert at the beginning (index 0) — most expensive, shifts everything
nums.insert(0, 5)
print(f"insert(0, 5):              {nums}")

# Insert at the end — equivalent to append
nums.insert(len(nums), 55)
print(f"insert(len, 55):           {nums}")

# Edge case: index beyond list length (CPython clamps to len)
nums.insert(100, 999)
print(f"insert(100, 999):          {nums}  ← appended at end")

# Edge case: negative index
test = [1, 2, 3]
test.insert(-1, 99)
print(f"insert(-1, 99):            {test}  ← inserted BEFORE last element")



### 2.4 `list.remove(x)`
- **What it does**: Removes the **first occurrence** of value `x`.
- **Returns**: `None`.
- **Raises**: `ValueError` if `x` is not in the list.
- **CPython**: Linear scan from index 0, comparing with `==` (not `is`). After finding, shifts all subsequent elements left.
- **Time**: $O(n)$ — scan + shift.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.remove(x)")
print("=" * 70)

items = [1, 2, 3, 2, 4, 2]
items.remove(2)  # Removes FIRST 2 only
print(f"remove(2) on [1,2,3,2,4,2]: {items}")

# Edge case: removing the only element
single = [42]
single.remove(42)
print(f"remove only element:         {single}")

# Edge case: ValueError when item not found
try:
    [1, 2, 3].remove(99)
except ValueError as e:
    print(f"remove(99) raises:           ValueError: {e}")

# Edge case: remove with custom objects (uses __eq__)
class Item:
    def __init__(self, v):
        self.v = v
    def __eq__(self, other):
        return isinstance(other, Item) and self.v == other.v
    def __repr__(self):
        return f"Item({self.v})"

objs = [Item(1), Item(2), Item(3)]
objs.remove(Item(2))  # Uses __eq__ for comparison
print(f"remove with __eq__:          {objs}")



### 2.5 `list.pop([i])`
- **What it does**: Removes and **returns** the element at index `i`. Default `i = -1` (last element).
- **Returns**: The removed element.
- **Raises**: `IndexError` if the list is empty or index is out of range.
- **CPython**: For `pop(-1)`, simply decrements `ob_size` — $O(1)$. For `pop(i)`, shifts all elements after `i` left — $O(n)$.
- **Time**: $O(1)$ for `pop()` / `pop(-1)`. $O(n)$ for `pop(i)` where `i < len-1`.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.pop([i])")
print("=" * 70)

stack = [10, 20, 30, 40, 50]
last = stack.pop()
print(f"pop() returns:               {last}, list now: {stack}")

first = stack.pop(0)
print(f"pop(0) returns:              {first}, list now: {stack}")

middle = stack.pop(1)
print(f"pop(1) returns:              {middle}, list now: {stack}")

# Edge case: pop from empty list
try:
    [].pop()
except IndexError as e:
    print(f"pop() on empty:              IndexError: {e}")

# Edge case: negative index
vals = [1, 2, 3, 4, 5]
second_last = vals.pop(-2)
print(f"pop(-2) returns:             {second_last}, list now: {vals}")



### 2.6 `list.clear()`
- **What it does**: Removes all elements. Equivalent to `del lst[:]`.
- **Returns**: `None`.
- **CPython**: Sets `ob_size = 0` and deallocates the items array.
- **Time**: $O(n)$ — must decrement reference counts for all elements.
- **Space**: Releases the pointer array memory.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.clear()")
print("=" * 70)

data = [1, 2, 3, 4, 5]
mem_before = sys.getsizeof(data)
data.clear()
mem_after = sys.getsizeof(data)
print(f"After clear():               {data}")
print(f"Memory before: {mem_before} bytes → after: {mem_after} bytes")

# clear vs reassignment
ref = [1, 2, 3]
alias = ref  # alias points to same object
ref.clear()  # Mutates the object in-place — alias also affected
print(f"ref after clear:             {ref}")
print(f"alias after ref.clear():     {alias}  ← also empty (same object)")



### 2.7 `list.index(x[, start[, end]])`
- **What it does**: Returns the index of the first occurrence of `x` in `list[start:end]`.
- **Returns**: `int`.
- **Raises**: `ValueError` if `x` is not found.
- **Time**: $O(n)$ — linear scan.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.index(x, start, end)")
print("=" * 70)

letters = ['a', 'b', 'c', 'b', 'd', 'b']
print(f"index('b'):                  {letters.index('b')}")
print(f"index('b', 2):               {letters.index('b', 2)}")  # Search from index 2
print(f"index('b', 2, 4):            {letters.index('b', 2, 4)}")  # Search [2:4]

# Edge case: not found
try:
    letters.index('z')
except ValueError as e:
    print(f"index('z') raises:           ValueError: {e}")



### 2.8 `list.count(x)`
- **What it does**: Returns the number of times `x` appears in the list.
- **Returns**: `int`.
- **Time**: $O(n)$ — must scan entire list.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.count(x)")
print("=" * 70)

nums = [1, 2, 3, 2, 2, 4, 1]
print(f"count(2):                    {nums.count(2)}")
print(f"count(5):                    {nums.count(5)}")  # Not present → 0
print(f"count(None) in [None,1,None]: {[None, 1, None].count(None)}")



### 2.9 `list.sort(*, key=None, reverse=False)`
- **What it does**: Sorts the list **in-place** using **Timsort** (hybrid merge sort + insertion sort).
- **Returns**: `None` (in-place). Use `sorted()` for a new sorted list.
- **CPython**: Timsort detects pre-existing ordered runs and merges them, achieving best-case $O(n)$ on nearly-sorted data.
- **Time**: $O(n \log n)$ average and worst case. $O(n)$ best case.
- **Space**: $O(n)$ for Timsort's temporary merge buffer.
- **Stability**: Timsort is a **stable** sort — equal elements preserve their original relative order.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.sort(key=None, reverse=False)")
print("=" * 70)

# Basic sort
vals = [3, 1, 4, 1, 5, 9, 2, 6]
vals.sort()
print(f"sort():                      {vals}")

# Reverse sort
vals.sort(reverse=True)
print(f"sort(reverse=True):          {vals}")

# Sort with key function
words = ["banana", "apple", "cherry", "date"]
words.sort(key=len)
print(f"sort(key=len):               {words}")

# Stability demonstration
pairs = [(1, 'b'), (2, 'a'), (1, 'a'), (2, 'b')]
pairs.sort(key=lambda p: p[0])
print(f"Stable sort by [0]:          {pairs}")
# Items with key=1 retain original order: (1,'b') before (1,'a')

# Edge case: sorting mixed types raises TypeError in Python 3
try:
    [1, "a", 3].sort()
except TypeError as e:
    print(f"sort mixed types:            TypeError: {e}")



### 2.10 `list.reverse()`
- **What it does**: Reverses the list **in-place**.
- **Returns**: `None`.
- **CPython**: Swaps pointers from both ends moving inward: `ob_item[i] ↔ ob_item[n-1-i]`.
- **Time**: $O(n)$ — swaps $n/2$ pairs.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.reverse()")
print("=" * 70)

seq = [1, 2, 3, 4, 5]
seq.reverse()
print(f"reverse():                   {seq}")

# reverse vs slicing [::-1]
original = [10, 20, 30]
reversed_copy = original[::-1]  # Creates a NEW list
original.reverse()               # Mutates in-place
print(f"[::-1] new list:             {reversed_copy}")
print(f".reverse() in-place:         {original}")
print(f"Same object? {reversed_copy is original}")  # False



### 2.11 `list.copy()`
- **What it does**: Returns a **shallow copy** of the list.
- **Returns**: A new `list` object.
- **CPython**: Allocates a new `PyListObject` and copies the `ob_item` pointer array. The pointed-to objects are NOT duplicated.
- **Time**: $O(n)$.
- **Shallow vs Deep**: Nested mutable objects (lists, dicts) inside the copy still reference the same objects.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: list.copy() — Shallow vs Deep Copy")
print("=" * 70)

import copy

original = [[1, 2], [3, 4], [5, 6]]

# Shallow copy
shallow = original.copy()
shallow[0][0] = 999
print(f"Original after shallow mutation: {original}")
# original[0][0] is ALSO 999 because the inner list is shared!

# Reset
original = [[1, 2], [3, 4], [5, 6]]

# Deep copy
deep = copy.deepcopy(original)
deep[0][0] = 999
print(f"Original after deep mutation:    {original}")
# original is unchanged because deepcopy cloned all nested objects

# Equivalences
lst = [1, 2, 3]
c1 = lst.copy()
c2 = lst[:]
c3 = list(lst)
print(f"copy() == [:] == list(): {c1 == c2 == c3}")  # True



---
## 3. SLICING INTERNALS

- **Syntax**: `lst[start:stop:step]`
- **CPython**: Slicing creates a **new list** containing copies of the pointers. It does NOT copy the underlying objects (shallow).
- The `__getitem__` method receives a `slice` object: `slice(start, stop, step)`.
- **Negative indices**: CPython converts `lst[-k]` to `lst[len(lst) - k]`.
- **Time**: $O(k)$ where $k$ = number of elements in the resulting slice.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: Slicing Internals")
print("=" * 70)

data = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

print(f"data[2:7]:                   {data[2:7]}")
print(f"data[::2]:                   {data[::2]}")     # Every 2nd element
print(f"data[::-1]:                  {data[::-1]}")     # Full reverse
print(f"data[7:2:-1]:                {data[7:2:-1]}")   # Reverse slice
print(f"data[-3:]:                   {data[-3:]}")      # Last 3

# Slice assignment (in-place mutation)
data[2:5] = [20, 30, 40, 50]  # Can insert more elements than replaced
print(f"After data[2:5]=[20..50]:    {data}")

# Delete via slice
del data[1:3]
print(f"After del data[1:3]:         {data}")

# Edge case: out-of-bounds slicing does NOT raise an error
print(f"data[100:200]:               {data[100:200]}")  # Returns empty list

# Memory: slicing creates a new list
orig = [1, 2, 3]
sliced = orig[:]
print(f"orig is sliced:              {orig is sliced}")  # False — different object



---
## 4. LIST COMPREHENSIONS INTERNALS

- **Syntax**: `[expr for item in iterable if condition]`
- **CPython**: Compiles to a special `LIST_APPEND` bytecode that bypasses the method dispatch overhead of calling `.append()` directly. This makes list comprehensions faster than equivalent `for` loops with `.append()`.
- **Nested comprehensions**: `[[expr for j in ...] for i in ...]` creates a list of lists.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: List Comprehension Internals & Performance")
print("=" * 70)

import time

N = 100_000

# Method 1: for loop with append
t0 = time.perf_counter()
result_loop = []
for i in range(N):
    result_loop.append(i * i)
t1 = time.perf_counter()
loop_time = (t1 - t0) * 1000

# Method 2: list comprehension
t0 = time.perf_counter()
result_comp = [i * i for i in range(N)]
t1 = time.perf_counter()
comp_time = (t1 - t0) * 1000

print(f"Loop with append:            {loop_time:.2f} ms")
print(f"List comprehension:          {comp_time:.2f} ms")
print(f"Speedup:                     {loop_time / (comp_time + 0.001):.1f}x")

# Nested comprehension
matrix = [[i * j for j in range(4)] for i in range(3)]
print(f"Nested comprehension:        {matrix}")

# Conditional comprehension
evens = [x for x in range(20) if x % 2 == 0]
print(f"Filter even numbers:         {evens}")



---
## 5. ITERATOR BEHAVIOR & GARBAGE COLLECTION

- `iter(lst)` returns a `list_iterator` object with `__next__`.
- **Mutation during iteration**: Modifying a list while iterating over it produces undefined behavior (elements may be skipped or duplicated). CPython does NOT raise an error but the results are unpredictable.
- **Garbage Collection**: When a list is deleted or goes out of scope, CPython decrements the reference count of every element. If any element's refcount reaches 0, it is deallocated immediately (reference counting). Circular references are handled by the cyclic garbage collector.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Iterator Behavior & GC Interaction")
print("=" * 70)

# Basic iteration protocol
nums = [10, 20, 30]
it = iter(nums)
print(f"next(iter): {next(it)}, {next(it)}, {next(it)}")
try:
    next(it)
except StopIteration:
    print("StopIteration raised after exhaustion")

# Dangerous: modifying list during iteration
print("\nDangerous mutation during iteration:")
data = [1, 2, 3, 4, 5]
for item in data:
    if item == 2:
        data.remove(2)  # Modifies list mid-iteration!
print(f"Result after removing 2 during iteration: {data}")
# Note: element 3 was SKIPPED because indices shifted

# Safe pattern: iterate over a copy
data = [1, 2, 3, 4, 5]
for item in data[:]:  # Iterate over a shallow copy
    if item == 2:
        data.remove(2)
print(f"Safe removal via copy:       {data}")

# Reference counting demo
print("\nReference counting:")
a = [1, 2, 3]
print(f"refcount of a: {sys.getrefcount(a)}")  # Usually 2 (variable + getrefcount arg)
b = a  # Another reference
print(f"refcount after b=a: {sys.getrefcount(a)}")  # 3
del b
print(f"refcount after del b: {sys.getrefcount(a)}")  # Back to 2



---
## 6. FAANG & RESEARCH INTERVIEW QUESTIONS

### Q1: What is the time complexity of `list.insert(0, x)` and why is it $O(n)$?
**A**: Inserting at index 0 requires shifting every existing pointer in the `ob_item` array one position to the right (via `memmove`). With $n$ elements, this is $O(n)$. For $O(1)$ front insertion, use `collections.deque`.

### Q2: Why does `lst.sort()` return `None` instead of the sorted list?
**A**: Python's convention for in-place mutations is to return `None` to signal that the original object was modified and no new object was created. This prevents confusion about whether the returned value is a new list or the same one. Use `sorted(lst)` for a new sorted list.

### Q3: Explain the difference between `lst.copy()`, `lst[:]`, `list(lst)`, and `copy.deepcopy(lst)`.
**A**: The first three all produce **shallow copies** — they create a new list object but the elements inside are the same references. `copy.deepcopy()` recursively clones every nested mutable object, producing a fully independent copy. For flat lists of immutables, all four behave identically.

### Q4: Why is iterating over a list and removing elements simultaneously dangerous?
**A**: The iterator uses an internal index counter. When you remove an element, all subsequent elements shift left by one. The iterator's index advances past the shifted element, skipping it. This leads to silent bugs. The safe pattern is to iterate over a copy (`lst[:]`) or use list comprehension to filter.

### Q5: What happens when you do `lst *= 3` with nested lists?
**A**: `lst *= 3` creates a list with 3 shallow copies of the original elements. If the list contains mutable objects (like inner lists), all three copies reference the **same** objects. Mutating one inner list mutates all of them. This is the classic "list multiplication trap".


In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: List Multiplication Trap Demo")
print("=" * 70)

# The trap
grid = [[0] * 3] * 3  # All 3 rows are THE SAME list object!
grid[0][0] = 1
print(f"grid after grid[0][0]=1:     {grid}")
print("All rows mutated because they are the same object!")

# The fix
grid_fixed = [[0] * 3 for _ in range(3)]  # Each row is a NEW list
grid_fixed[0][0] = 1
print(f"Fixed grid:                  {grid_fixed}")
print("Only row 0 was mutated.")

# Verify identity
bad = [[0]] * 3
print(f"\nbad[0] is bad[1]: {bad[0] is bad[1]}")  # True — same object
good = [[0] for _ in range(3)]
print(f"good[0] is good[1]: {good[0] is good[1]}")  # False — different objects
